# Option Skew Risk-Reversal Strategy (Sondre)

Implements the strategy from `option_skew_strategy.py` using the production pipeline.

## Strategy overview
1. Fit SSVI vol surface per ticker/date.
2. Compute daily 25-delta risk-reversal skew at a ~15-day tenor.
3. Regress each stock's skew on the sector ETF (XLF) with a 60-day rolling OLS window.
4. Z-score the residual (Gaussian, 60-day window). Enter when |z| > 1.0, exit when |z| < 0.3.
5. Trade a delta-hedged risk reversal (OTM put + OTM call at ~25 delta, ~15 dte).
   - z > +entry → **short skew** (sell the risk reversal: buy put, sell call)
   - z < −entry → **long skew** (buy the risk reversal: buy call, sell put)
6. Daily PnL = option-leg PnL + delta-hedge PnL − transaction costs.

In [ ]:
import sys
import os


# Add pipeline root to path
PIPELINE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mpl_ticker
print("Imports OK")

Imports OK


In [ ]:
from pipeline import Pipeline, MasterRunSpec, DataSpec, VolatilityModelSpec, SkewSpec, SignalSpec, BacktestSpec
from analytics.signal_models import plot_signal_analytics, summarize_signals, trade_log
from analytics.backtest_models import plot_backtest_analytics, summarize_backtest, pnl_attribution

## Configuration

Parameters match the original `option_skew_strategy.py`:
- Tickers: XLF (sector benchmark), GS, JPM, BAC, C
- Skew: 25-delta risk reversal, 15-day trade tenor
- Signal: 60-day rolling OLS residual z-score, entry |z| > 1.0, exit |z| < 0.3
- Backtest: delta-hedged risk reversal, $0.65/contract fee, 1 bps stock fee

In [ ]:
cfg = MasterRunSpec(
    data=DataSpec(
        tickers=["XLF", "GS", "JPM", "BAC", "C"],
        start_date="2008-01-01",
        end_date="2015-12-31",
    ),
    model=VolatilityModelSpec(
        model_kind="surface",
        ssvi_n_theta_steps=3,
        ssvi_n_restarts=4,
    ),
    skew=SkewSpec(
        technique="rr_approx",
        rr_delta=0.25,          # 25-delta RR (matches original delta_target=0.25)
        rr_k=0.15,              # fallback log-moneyness width
        tenor_days=[15, 30],    # 15d = trade tenor; 30d used for term-structure event filter
    ),
    signal=SignalSpec(
        model_kind="risk_reversal",
        regression_window=60,   # matches estimation_window=60 in original
        min_regression_obs=40,
        zscore_window=60,       # matches signal_window=60 in original
        entry_z=1.0,            # matches entry_threshold in original
        exit_z=0.3,             # matches exit_threshold in original
        min_hold_days=1,
        min_liquidity_points=20,
        max_fit_rmse_iv=0.50,
        direction="mean_revert",
        cross_section_top_k=2,
        cross_section_bottom_k=2,
        max_gross_leverage=1.5,
        benchmark_preference="XLF",
        trade_tenor_days=15,    # matches tte_target=15 in original
        ts_event_threshold=2.0,
        event_filter_enabled=True,
        zscore_method="gaussian",
    ),
    backtest=BacktestSpec(
        stock_fee_bps=1.0,
        option_fee_per_contract=0.65,
        sizing_mode="contracts",
        max_contracts_per_signal=10.0,
        target_dollar_vega_per_signal=20000.0,
        roll_threshold_days=5,
    ),
)

p = Pipeline(cfg)
print("Pipeline built OK")
print(f"Tickers       : {cfg.data.tickers}")
print(f"Date range    : {cfg.data.start_date} -> {cfg.data.end_date}")
print(f"Trade tenor   : {cfg.signal.trade_tenor_days}d  delta target: {cfg.skew.rr_delta}")
print(f"Entry z / Exit z: {cfg.signal.entry_z} / {cfg.signal.exit_z}")

## Pipeline stages 1–3: Load → Process → Model

These stages are shared with `back_to_basics.ipynb` and use the `basics.parquet` cache.

In [ ]:
loaded    = p.run_load(saved="basics.parquet")
print(f"Loaded   : {list(loaded.raw_by_ticker.keys())}")

processed = p.run_process(loaded, saved="basics.parquet")
print(f"Processed: {list(processed.panel_by_ticker.keys())}")

modeled   = p.run_model(processed, saved="basics.parquet")
print(f"Model    : {modeled.representation}  tickers={list(modeled.model_by_ticker.keys())}")

## Stage 4: Skew computation

Computes the 25-delta risk-reversal skew at 15-day and 30-day tenors using the `rr_approx` method.

In [ ]:
skew = p.run_skew(modeled, saved="sondre_skew.parquet")

print(f"Skew tickers : {list(skew.skew_by_ticker.keys())}")
sample = next(iter(skew.skew_by_ticker.values()))
print(f"Skew df shape: {sample.shape}  columns: {list(sample.columns)}")
print(sample.head(6).to_string())

In [ ]:
# Quick summary: skew stats at 15d tenor
parts = []
for ticker, df in skew.skew_by_ticker.items():
    if df is None or df.empty:
        continue
    x = df.copy()
    x["ticker"] = ticker
    x["date"] = pd.to_datetime(x["date"])
    parts.append(x)
skew_df = pd.concat(parts, ignore_index=True).sort_values(["ticker", "tenor_days", "date"]).reset_index(drop=True)

td15 = skew_df[skew_df["tenor_days"] == 15]
print("15d skew summary:")
print(
    td15.groupby("ticker")["skew"]
    .agg(["mean", "std", "min", "max", "count"])
    .round(4)
    .to_string()
)

## Stage 5: Signal generation

Rolling OLS residual z-score vs XLF benchmark at the 15-day tenor.

- **+position (long RR)**: z < −1.0 → idiosyncratic skew cheap vs sector → buy risk reversal
- **−position (short RR)**: z > +1.0 → idiosyncratic skew rich vs sector → sell risk reversal
- **flat**: |z| < 0.3 after holding

In [ ]:
signals = p.run_signals(skew)

print(f"Signal tickers: {list(signals.signal_map.keys())}")
print()
print(f"{'ticker':<8}  {'long':>6}  {'short':>6}  {'flat':>6}  {'total':>6}")
for ticker, df in signals.signal_map.items():
    if df is None or df.empty:
        continue
    pos = df["position"].dropna()
    n_long  = int((pos >  0.01).sum())
    n_short = int((pos < -0.01).sum())
    n_flat  = int((pos.abs() <= 0.01).sum())
    print(f"{ticker:<8}  {n_long:>6}  {n_short:>6}  {n_flat:>6}  {len(pos):>6}")

In [ ]:
# Three-panel signal plot: residual, z-score with thresholds, position
try:
    log = plot_signal_analytics(signals, engine="plotly")
except Exception:
    log = plot_signal_analytics(signals, engine="matplotlib")
    plt.show()

In [ ]:
# Trade log and per-ticker signal summary
print("=== Per-ticker signal summary ===")
print(summarize_signals(signals).to_string(index=False))

print("\n=== Trade log (first 40 events) ===")
tlog = trade_log(signals)
print(tlog.head(40).to_string(index=False))

## Stage 6: Backtest

Simulates delta-hedged risk-reversal positions using the pipeline's `DeltaHedgedOptionBacktestEngine`:
- Each day, select OTM call + OTM put closest to 25-delta and ~15 dte.
- Daily PnL = option mid-price change + daily delta-hedge PnL.
- Costs: $0.65/contract + 1 bps on hedge trades.
- Positions are sized by `max_contracts_per_signal=10`.

In [ ]:
backtest = p.run_backtest(signals, skew)

print(f"Portfolio rows   : {len(backtest.portfolio)}")
print(f"Portfolio columns: {list(backtest.portfolio.columns)}")
print()
print(backtest.portfolio.tail(8).to_string())

In [ ]:
print("=== Backtest Summary ===")
bt_summary = summarize_backtest(backtest)
print(bt_summary.T.to_string())

print("\n=== PnL Attribution by Ticker ===")
print(pnl_attribution(backtest).to_string(index=False))

print("\n=== Pipeline summary dict ===")
for k, v in sorted(backtest.summary.items()):
    print(f"  {k:<42} {v:.6f}")

In [ ]:
# Full backtest dashboard
try:
    plot_backtest_analytics(backtest, engine="plotly")
except Exception:
    plot_backtest_analytics(backtest, engine="matplotlib")
    plt.show()

## Manual plots (mirrors `option_skew_strategy.plot_results`)

In [ ]:
p_df = backtest.portfolio.copy()
p_df["date"] = pd.to_datetime(p_df["date"])
p_df = p_df.sort_values("date").reset_index(drop=True)

# 1. Cumulative PnL decomposition
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(p_df["date"], p_df["portfolio_cum_net_pnl"], linewidth=1.8, label="Cumulative net PnL")
if "portfolio_option_pnl" in p_df.columns:
    ax.plot(p_df["date"], p_df["portfolio_option_pnl"].cumsum(),
            linewidth=1.2, linestyle="--", label="Option legs")
if "portfolio_hedge_pnl" in p_df.columns:
    ax.plot(p_df["date"], p_df["portfolio_hedge_pnl"].cumsum(),
            linewidth=1.2, linestyle=":", label="Delta hedge")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Cumulative PnL — 25-Delta Risk Reversal Strategy (15-day tenor, vs XLF)")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative PnL ($)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 2. Drawdown
cum = p_df["portfolio_cum_net_pnl"]
drawdown = cum - cum.cummax()
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(p_df["date"], drawdown, color="red", linewidth=1)
ax.fill_between(p_df["date"], drawdown, 0, alpha=0.25, color="red")
ax.set_title("Drawdown (Net PnL)")
ax.set_xlabel("Date")
ax.set_ylabel("Drawdown ($)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 3. Annual net PnL
annual = p_df.set_index("date")["portfolio_net_pnl"].resample("YE").sum()
annual.index = annual.index.year

colors = ["green" if v > 0 else "red" for v in annual]
fig, ax = plt.subplots(figsize=(max(6, len(annual) * 1.2), 4))
annual.plot(kind="bar", ax=ax, color=colors, edgecolor="black")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Annual Net PnL")
ax.set_xlabel("Year")
ax.set_ylabel("Net PnL ($)")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

print("Annual PnL:")
print(annual.to_string())

## Parameter sensitivity: entry z-score

Sweeps entry thresholds to assess robustness. Reuses the cached skew output.

In [ ]:
from dataclasses import replace

entry_z_values = [0.75, 1.0, 1.25, 1.5, 2.0]
sweep_rows = []

for entry_z in entry_z_values:
    cfg_sw = replace(cfg, signal=replace(cfg.signal, entry_z=entry_z))
    p_sw   = Pipeline(cfg_sw)
    sigs_sw = p_sw.run_signals(skew)
    bt_sw   = p_sw.run_backtest(sigs_sw, skew)
    s = bt_sw.summary
    sweep_rows.append({
        "entry_z":      entry_z,
        "sharpe":       s.get("daily_sharpe_like",  float("nan")),
        "cum_net_pnl":  s.get("final_cum_net_pnl",  float("nan")),
        "max_drawdown": s.get("max_drawdown",        float("nan")),
        "hit_rate":     s.get("hit_rate",            float("nan")),
    })

sweep_df = pd.DataFrame(sweep_rows)
print("Entry z-score sensitivity:")
print(sweep_df.round(4).to_string(index=False))